# 2 - Preparing the Occupancy Data

**Purpose:** This notebook serves to prepare the data for modelling, which includes seperating data into appropriate splits (training, validating and testing). The information for each forecast includes current headcount, headcounts 15 and 30 minutes earlier, time of day and day of the week.

**Splits:**
| Splits | Period |
|---|---|
| Training | March 2024–August 2025 |
| Validation | September–December 2025 |
| Final testing | January–March 2026 |

### **Part 1:**
Extracting and checking raw records




In [7]:
from pathlib import Path
import zipfile
import pandas as pd
from IPython.display import display

print("pandas version:", pd.__version__)

pandas version: 3.0.6


### Identify the Data and Chosen Building space
Decided to use the space (227f1068) as seen from the occupancy feasibility notebook, it offeres one of the highest usable percentage of data along with the highest variation in headcount.

In [8]:

zip_path = Path("../data/DataSet2.zip")

csv_name = (
    "DataSet2_Env_and_Occupancy/"
    "5occupancySensor_MayToDec2024_9MRows/"
    "5occupancySensor_MayToDec2024_9MRows.csv"
)


# Select the full identifier so a shortened display label cannot match the wrong space.
space_id = "227f1068-9e41-469f-bef0-275f27a0bdba"

print("Working folder:", Path.cwd())
print("ZIP found:", zip_path.exists())
print("Selected space:", space_id)

Working folder: c:\Users\chris\OneDrive\Desktop\MMA3001\Project\mma3001_project\notebooks
ZIP found: True
Selected space: 227f1068-9e41-469f-bef0-275f27a0bdba


### Recording/Defining the chosen splits
Splits were chosen to be chronological (considering the whole idea is to use the past to predict the future). Additionally, each period includes its start but excludes its end

In [9]:
# Fix the chronological periods before fitting models or inspecting test errors.
train_start = pd.Timestamp("2024-03-01", tz="Australia/Melbourne")
validation_start = pd.Timestamp("2025-09-01", tz="Australia/Melbourne")
test_start = pd.Timestamp("2026-01-01", tz="Australia/Melbourne")
study_end = pd.Timestamp("2026-04-01", tz="Australia/Melbourne")

horizon = pd.Timedelta(minutes=30)
# Limit how old a reading can be when representing a requested headcount.
max_age = pd.Timedelta(minutes=5)
forecast_frequency = "15min"

print("Study starts:", train_start)
print("Validation starts:", validation_start)
print("Testing starts:", test_start)
print("Study ends (excluded):", study_end)

Study starts: 2024-03-01 00:00:00+11:00
Validation starts: 2025-09-01 00:00:00+10:00
Testing starts: 2026-01-01 00:00:00+11:00
Study ends (excluded): 2026-04-01 00:00:00+11:00


### Organising Data set to only include chosen space
Just filtering data to only include the rows containing the correct space

In [10]:
columns = ["floorspaceid", "collecteddate", "headcount", "occupancystatus"]
space_parts = []

with zipfile.ZipFile(zip_path) as zip_file:
    with zip_file.open(csv_name) as csv_file:
        # Process batches so the complete history for all spaces need not stay in memory.
        for chunk in pd.read_csv(csv_file, usecols=columns, chunksize=200000):
            # Keep one monitored space so its history is not mixed with other occupancy patterns.
            selected_rows = chunk[chunk["floorspaceid"] == space_id]
            space_parts.append(selected_rows)

raw_space = pd.concat(space_parts, ignore_index=True)

print("Raw records for the selected space:", len(raw_space))
print("Number of space IDs:", raw_space["floorspaceid"].nunique())

Raw records for the selected space: 1500394
Number of space IDs: 1


#### Converts Dates and heacounts, then checks them

In [11]:
# Compare instants in UTC while retaining Melbourne time for the model clock inputs.
raw_space["timestamp"] = pd.to_datetime(
    raw_space["collecteddate"],
    format="ISO8601",
    utc=True,
    errors="coerce"
)

raw_space["local_time"] = raw_space["timestamp"].dt.tz_convert(
    "Australia/Melbourne"
)

# Make unreadable counts visible as missing values so the checks below can stop.
raw_space["headcount"] = pd.to_numeric(
    raw_space["headcount"], errors="coerce"
)

invalid_times = raw_space["timestamp"].isna().sum()
invalid_counts = raw_space["headcount"].isna().sum()
negative_counts = (raw_space["headcount"] < 0).sum()
non_integer_counts = (raw_space["headcount"].dropna() % 1 != 0).sum()

print("Invalid timestamps:", invalid_times)
print("Missing or non-numeric headcounts:", invalid_counts)
print("Negative headcounts:", negative_counts)
print("Non-integer or non-finite headcounts:", non_integer_counts)

# Stop on unexpected data rather than quietly choosing a repair rule.
assert len(raw_space) > 0, "No rows found. Check the space ID."
assert invalid_times == 0, "Stop: inspect invalid timestamps before continuing."
assert invalid_counts == 0, "Stop: inspect invalid headcounts before continuing."
assert negative_counts == 0, "Stop: inspect negative headcounts before continuing."
assert non_integer_counts == 0, "Stop: inspect non-integer headcounts before continuing."

Invalid timestamps: 0
Missing or non-numeric headcounts: 0
Negative headcounts: 0
Non-integer or non-finite headcounts: 0


### Keep the study period and Inspect repeated timestamps

In [12]:
# Retain only the chosen study period without changing the source dataset.
space_data = raw_space[
    (raw_space["local_time"] >= train_start)
    & (raw_space["local_time"] < study_end)
].copy()

space_data = space_data.sort_values("timestamp")
space_data = space_data.reset_index(drop=True)

print("Records in the study period:", len(space_data))
print("First observation:", space_data["local_time"].min())
print("Last observation:", space_data["local_time"].max())
print("Extra rows at repeated timestamps:", space_data["timestamp"].duplicated().sum())

print("First five observations:")
display(space_data[["local_time", "headcount"]].head())

# Display every record at a repeated time so disagreements cannot be hidden.
repeated_times = space_data[
    space_data["timestamp"].duplicated(keep=False)
]

print("All records involved in repeated timestamps:")
display(repeated_times[["local_time", "headcount", "occupancystatus"]])

Records in the study period: 1483464
First observation: 2024-03-03 15:06:30+11:00
Last observation: 2026-03-31 23:17:52+11:00
Extra rows at repeated timestamps: 2
First five observations:


,local_time,headcount
0,2024-03-03 15:06:30+11:00,0
1,2024-03-03 15:08:18+11:00,1
2,2024-03-03 15:08:50+11:00,0
3,2024-03-03 15:09:02+11:00,1
4,2024-03-03 15:09:42+11:00,0


All records involved in repeated timestamps:


,local_time,headcount,occupancystatus
56782,2024-04-05 09:13:42+11:00,0,RecentlyOccupied
56783,2024-04-05 09:13:42+11:00,0,NotOccupied
208917,2024-05-23 09:19:28+10:00,0,RecentlyOccupied
208918,2024-05-23 09:19:28+10:00,0,NotOccupied


### **Part 2:**
Turns extracted sections into forecasting examples, aach example is one forecast made at time **t**. Its inputs are the headcounts near **t**, **t − 15 minutes** and **t − 30 minutes**, plus Melbourne time of day and day of week. Its target is the sensor reported headcount near **t + 30 minutes**. It is **not** the average headcount over the next 30 minutes.

Every required count uses the most recent observation at or before its requested time, no more than five minutes old. A missing match stays missing, it is not treated as zero.

### Give each measurment time one headcount
Two repeated timestamps were identified above. The headcounts agree, although the occupancy-status labels differ. For a headcount series, retaining one copy therefore preserves the recorded count.

The code first checks **every repeated timestamp for conflicting counts**. It stops if it finds a disagreement rather than choosing a value silently. It then creates a separate headcount series, leaving `space_data` and the original ZIP unchanged.

In [13]:
# Confirm that repeated counts agree before keeping a single observation.
for repeated_time in repeated_times["timestamp"].unique():
    matching_rows = repeated_times[
        repeated_times["timestamp"] == repeated_time
    ]
    assert matching_rows["headcount"].nunique() == 1, (
        "Conflicting headcounts at a repeated timestamp. Investigate before continuing."
    )

# Keep a separate count series so the original status records remain available.
headcount_data = space_data[["timestamp", "headcount"]].drop_duplicates(
    subset="timestamp", keep="first"
)
headcount_data = headcount_data.sort_values("timestamp")

readings = headcount_data.set_index("timestamp")["headcount"]
# Retain observation times to check the age of each matched reading later.
recorded_times = pd.Series(readings.index, index=readings.index)

assert readings.index.is_unique, "Repeated timestamps remain."
assert readings.index.is_monotonic_increasing, "Readings must be in time order."

print("Headcount observations after resolving repeats:", len(readings))
print("Repeated rows removed:", len(space_data) - len(readings))

Headcount observations after resolving repeats: 1483462
Repeated rows removed: 2


### Creating the forecasting schedule
Create a forecast starting time every 15 minutes throughout the study period. Only keep starts whose 30-minute target is before the end of the study.

This is a schedule, not a set of predictions or a claim that data is available at every time. The timestamps retain their timezone, so a 30-minute horizon means 30 minutes of elapsed time, including across daylight-saving transitions.

In [14]:
# Even the oldest accepted target must genuinely come after the forecast begins.
assert horizon > max_age, "The target observation must occur after the forecast starts."

# A regular schedule prevents frequent sensor reporting from creating extra forecast opportunities.
forecast_times = pd.date_range(
    start=train_start,
    end=study_end,
    freq=forecast_frequency,
    inclusive="left"
)
# Exclude starts whose target would fall outside the study.
forecast_times = forecast_times[forecast_times + horizon < study_end]

examples = pd.DataFrame({
    "forecast_time": forecast_times,
    "target_time": forecast_times + horizon
})

print("Scheduled forecasting examples:", len(examples))
print("First forecast time:", examples["forecast_time"].iloc[0])
print("Last target time:", examples["target_time"].iloc[-1])

Scheduled forecasting examples: 73054
First forecast time: 2024-03-01 00:00:00+11:00
Last target time: 2026-03-31 23:45:00+11:00


### Find the four required headcounts

The loop below makes four lookups: now, 15 minutes earlier, 30 minutes earlier and 30 minutes later.

In [15]:
lookup_offsets = {
    "headcount_now": pd.Timedelta(minutes=0),
    "headcount_lag15": pd.Timedelta(minutes=-15),
    "headcount_lag30": pd.Timedelta(minutes=-30),
    "target_headcount": horizon
}

for column, offset in lookup_offsets.items():
    requested_times = (forecast_times + offset).tz_convert("UTC")

    # Look backwards only and leave a gap when the latest reading is too old.
    matched_counts = readings.reindex(
        requested_times, method="ffill", tolerance=max_age
    )
    # Save the actual observation times so the timing assumptions can be checked.
    matched_times = recorded_times.reindex(
        requested_times, method="ffill", tolerance=max_age
    )

    # Keep values in forecast order rather than aligning the different timestamp indexes.
    examples[column] = matched_counts.to_numpy()
    examples[column + "_observed_at"] = pd.to_datetime(
        matched_times.to_numpy(), utc=True
    )
    examples[column + "_age_minutes"] = (
        (examples["forecast_time"] + offset)
        - examples[column + "_observed_at"]
    ).dt.total_seconds() / 60

print("Headcount lookups completed.")
print("Missing matches for each required count:")
display(examples[list(lookup_offsets)].isna().sum().to_frame("missing_matches"))

Headcount lookups completed.
Missing matches for each required count:


,missing_matches
headcount_now,34594
headcount_lag15,34595
headcount_lag30,34595
target_headcount,34594


### Add time inputs and keep complete examples

In [16]:
# Use the scheduled forecast time, not the age of the latest sensor reading.
examples["time_of_day"] = (
    examples["forecast_time"].dt.hour
    + examples["forecast_time"].dt.minute / 60
)
examples["day_of_week"] = examples["forecast_time"].dt.dayofweek

# List model inputs explicitly so future targets and audit fields stay out.
feature_columns = [
    "headcount_now", "headcount_lag15", "headcount_lag30",
    "time_of_day", "day_of_week"
]
target_column = "target_headcount"
count_columns = list(lookup_offsets)

# Measure how many examples are lost when the two historical counts are required.
current_target_available = examples[
    ["headcount_now", target_column]
].notna().all(axis=1)

# Require recorded counts rather than inventing missing inputs or future targets.
model_data = examples.dropna(subset=count_columns).copy()
model_data = model_data.reset_index(drop=True)

print("Scheduled examples:", len(examples))
print("Examples with current count and target:", int(current_target_available.sum()))
print("Examples also containing both historical counts:", len(model_data))
print("Extra examples lost when requiring history:",
      int(current_target_available.sum()) - len(model_data))
print("Complete-example availability (%):", round(100 * len(model_data) / len(examples), 2))

# Preview early training rows without displaying final test targets.
training_preview = model_data[model_data["target_time"] < validation_start]
display(training_preview[
    ["forecast_time"] + feature_columns + [target_column]
].head())

Scheduled examples: 73054
Examples with current count and target: 28614
Examples also containing both historical counts: 20421
Extra examples lost when requiring history: 8193
Complete-example availability (%): 27.95


,forecast_time,headcount_now,headcount_lag15,headcount_lag30,time_of_day,day_of_week,target_headcount
0,2024-03-03 16:00:00+11:00,1.0,1.0,1.0,16.00,6,1.0
1,2024-03-03 17:00:00+11:00,1.0,0.0,1.0,17.00,6,0.0
2,2024-03-03 17:30:00+11:00,0.0,0.0,1.0,17.50,6,1.0
3,2024-03-04 09:00:00+11:00,1.0,0.0,1.0,9.00,0,4.0
4,2024-03-04 09:15:00+11:00,1.0,1.0,0.0,9.25,0,0.0


### Applying the chronological splits

In [17]:
# Both the forecast and its target must belong to the same time period.
train_data = model_data[
    (model_data["forecast_time"] >= train_start)
    & (model_data["forecast_time"] < validation_start)
    & (model_data["target_time"] < validation_start)
].copy()

validation_data = model_data[
    (model_data["forecast_time"] >= validation_start)
    & (model_data["forecast_time"] < test_start)
    & (model_data["target_time"] < test_start)
].copy()

test_data = model_data[
    (model_data["forecast_time"] >= test_start)
    & (model_data["forecast_time"] < study_end)
    & (model_data["target_time"] < study_end)
].copy()

# Earlier history may support a later forecast because it was already available then.
split_data = {
    "train": train_data,
    "validation": validation_data,
    "test": test_data
}
split_periods = {
    "train": (train_start, validation_start),
    "validation": (validation_start, test_start),
    "test": (test_start, study_end)
}

summary_rows = []
for name, dataset in split_data.items():
    period_start, period_end = split_periods[name]
    # Use all eligible scheduled times as the denominator, including times with no usable data.
    within_period = (
        (examples["forecast_time"] >= period_start)
        & (examples["forecast_time"] < period_end)
        & (examples["target_time"] < period_end)
    )
    scheduled = int(within_period.sum())
    summary_rows.append({
        "split": name,
        "scheduled_within_period": scheduled,
        "usable_examples": len(dataset),
        "usable_percent": round(100 * len(dataset) / scheduled, 2),
        "first_forecast": dataset["forecast_time"].min(),
        "last_forecast": dataset["forecast_time"].max()
    })

split_summary = pd.DataFrame(summary_rows)
boundary_exclusions = len(model_data) - sum(len(data) for data in split_data.values())

print("Complete examples excluded because targets cross a split boundary:", boundary_exclusions)
display(split_summary)

Complete examples excluded because targets cross a split boundary: 1


,split,scheduled_within_period,usable_examples,usable_percent,first_forecast,last_forecast
0,train,52706,14772,28.03,2024-03-03 16:00:00+11:00,2025-08-31 23:00:00+10:00
1,validation,11706,3388,28.94,2025-09-01 00:30:00+10:00,2025-12-30 13:45:00+11:00
2,test,8638,2260,26.16,2026-01-01 08:15:00+11:00,2026-03-31 17:15:00+11:00


### Verifying the forecasting examples

In [18]:
# Check preparation rules before any model is allowed to learn from these examples.
for name, dataset in split_data.items():
    period_start, period_end = split_periods[name]

    assert len(dataset) > 0, f"No usable examples in {name}."
    assert dataset["forecast_time"].is_monotonic_increasing
    assert dataset["forecast_time"].is_unique
    assert dataset[feature_columns + [target_column]].notna().all().all()
    assert (dataset[count_columns] >= 0).all().all()
    assert (dataset[count_columns] % 1 == 0).all().all()
    assert (dataset["forecast_time"] >= period_start).all()
    assert (dataset["target_time"] < period_end).all()
    assert ((dataset["target_time"] - dataset["forecast_time"]) == horizon).all()

    # Check every historical count and the future reference against its own requested time.
    for column, offset in lookup_offsets.items():
        requested_time = dataset["forecast_time"] + offset
        observed_time = dataset[column + "_observed_at"]
        age = requested_time - observed_time
        assert age.notna().all(), f"Missing observation times in {name}: {column}"
        assert (age >= pd.Timedelta(0)).all(), "A reading comes from after its requested time."
        assert (age <= max_age).all(), "A reading exceeds the freshness allowance."

    assert (dataset["target_headcount_observed_at"] > dataset["forecast_time"]).all()
    print(f"{name}: preparation checks passed for {len(dataset):,} examples.")

# Prevent later evaluation targets from entering an earlier training period.
assert train_data["target_time"].max() < validation_data["forecast_time"].min()
assert validation_data["target_time"].max() < test_data["forecast_time"].min()
assert target_column not in feature_columns

print("All preparation checks passed. No regression model has been trained or tested.")

train: preparation checks passed for 14,772 examples.
validation: preparation checks passed for 3,388 examples.
test: preparation checks passed for 2,260 examples.
All preparation checks passed. No regression model has been trained or tested.


### Saving the datasets locally

In [19]:
import json

# Save one shared dataset so the models cannot accidentally use different examples.
output_folder = Path("../data/prepared") / space_id[:8]
output_folder.mkdir(parents=True, exist_ok=True)

time_columns = ["forecast_time", "target_time"] + [
    column + "_observed_at" for column in lookup_offsets
]

for name, dataset in split_data.items():
    exported_data = dataset.copy()
    # Store dates in UTC so CSV loading does not have to mix daylight saving offsets.
    for column in time_columns:
        exported_data[column] = exported_data[column].dt.tz_convert("UTC")
    exported_data.to_csv(output_folder / f"{name}.csv", index=False)
    print(f"Saved {name}.csv: {len(exported_data):,} examples")

split_summary.to_csv(output_folder / "split_summary.csv", index=False)

# Record the choices needed to reproduce these examples and check later model runs.
preparation_settings = {
    "space_id": space_id,
    "source_zip": zip_path.as_posix(),
    "source_csv": csv_name,
    "local_timezone": "Australia/Melbourne",
    "saved_timestamp_timezone": "UTC",
    "forecast_frequency": forecast_frequency,
    "horizon_minutes": horizon.total_seconds() / 60,
    "max_reading_age_minutes": max_age.total_seconds() / 60,
    "train_start": train_start.isoformat(),
    "validation_start": validation_start.isoformat(),
    "test_start": test_start.isoformat(),
    "study_end_exclusive": study_end.isoformat(),
    "feature_columns": feature_columns,
    "target_column": target_column,
    "timestamp_columns": time_columns,
    "time_of_day_units": "Decimal hours in Melbourne local time",
    "day_of_week_codes": "Monday=0 through Sunday=6; categorical labels",
    "missing_rule": "Require all three input counts and the target; no imputation",
    "lookup_rule": "Latest reading at or before each requested time, within max_age",
    "duplicate_rule": "Stop on conflicting counts; collapse agreeing counts at the same timestamp",
    "split_rule": "Forecast and nominal target must be in the same chronological partition",
    "test_use": "Reserved for final evaluation after model selection",
    "pandas_version": pd.__version__,
    "split_counts": {name: len(dataset) for name, dataset in split_data.items()}
}

with open(output_folder / "preparation_settings.json", "w", encoding="utf-8") as file:
    json.dump(preparation_settings, file, indent=2)

print("Prepared data folder:", output_folder.resolve())

Saved train.csv: 14,772 examples
Saved validation.csv: 3,388 examples
Saved test.csv: 2,260 examples
Prepared data folder: C:\Users\chris\OneDrive\Desktop\MMA3001\Project\mma3001_project\data\prepared\227f1068
